In [1]:
import numpy as np
import pandas as pd
import os
import scipy.io
from tensorflow import keras
from keras import layers
from keras.utils import load_img,np_utils, img_to_array, to_categorical
from sklearn.model_selection import train_test_split
from keras.applications.vgg16 import VGG16
from keras.models import Model
from keras.layers import Dense, Flatten, Dropout
from keras.optimizers import Adam
from sklearn.model_selection import train_test_split
#from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
image_folder = 'flowers/'
label_file = 'imagelabels.mat'

In [3]:
labels = scipy.io.loadmat(label_file)
data = labels['labels']

In [4]:
num_classes = np.max(data) + 1
data = to_categorical(data, num_classes=num_classes)


In [5]:
data = data.reshape(data.shape[1:])
data.shape

(8189, 103)

In [6]:
image_files = os.listdir(image_folder)
print(len(image_files))
files = [item for item in image_files if ')' not in item]
print(len(files))
files1 = [item for item in files if 'ipynb_checkpoints' not in item]
print(len(files1))

8189
8189
8189


In [9]:
image_files = os.listdir(image_folder)
print(len(files1))
images = []
for filename in files1:
  img = load_img(image_folder+"/"+filename, target_size=(128, 128))  # VGG16 expects input of size 224x224
  img_array = img_to_array(img)
  images1 = np.expand_dims(img_array, axis=0)
  images.append(images1)
images = np.array(images)
print(images.shape)

8189
(8189, 1, 128, 128, 3)


In [10]:
images /= 255.
images = images.reshape(8189, 128, 128, 3)
images.shape

(8189, 128, 128, 3)

In [11]:
train_images, test_images, train_labels, test_labels = train_test_split(images, data,train_size=0.8 ,test_size=0.2, random_state=42)

In [12]:
print(train_images.shape)
print(train_labels.shape)
print(test_images.shape)
print(test_labels.shape)

(6551, 128, 128, 3)
(6551, 103)
(1638, 128, 128, 3)
(1638, 103)


In [13]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# Make the layers in the base model non-trainable
for layer in base_model.layers:
    layer.trainable = False

# Add new layers on top for the classification
x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [14]:
model = Model(inputs=base_model.input, outputs=predictions)

In [15]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

In [16]:
model.fit(train_images, train_labels, epochs=40, validation_data=(test_images, test_labels))

Epoch 1/40
205/205 [==============================] - 478s 2s/step - loss: 4.4781 - accuracy: 0.0516 - val_loss: 4.1695 - val_accuracy: 0.1306
Epoch 2/40
205/205 [==============================] - 688s 3s/step - loss: 3.9980 - accuracy: 0.1421 - val_loss: 3.6798 - val_accuracy: 0.2613
Epoch 3/40
205/205 [==============================] - 697s 3s/step - loss: 3.5474 - accuracy: 0.2218 - val_loss: 3.2713 - val_accuracy: 0.3425
Epoch 4/40
205/205 [==============================] - 688s 3s/step - loss: 3.2198 - accuracy: 0.2810 - val_loss: 2.9509 - val_accuracy: 0.4109
Epoch 5/40
205/205 [==============================] - 1344s 7s/step - loss: 2.9131 - accuracy: 0.3433 - val_loss: 2.6820 - val_accuracy: 0.4603
Epoch 6/40
205/205 [==============================] - 687s 3s/step - loss: 2.6866 - accuracy: 0.3804 - val_loss: 2.4776 - val_accuracy: 0.5092
Epoch 7/40
205/205 [==============================] - 1272s 6s/step - loss: 2.4760 - accuracy: 0.4231 - val_loss: 2.3274 - val_accuracy: 0.52

In [17]:
loss, accuracy = model.evaluate(test_images, test_labels)
print(f"Test Accuracy: {accuracy * 100}%")

52/52 [==============================] - 134s 3s/step - loss: 1.0006 - accuracy: 0.7564
Test Accuracy: 75.64102411270142%
